In [1]:
import os
import sys

python_path = sys.executable

print(python_path)

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

c:\Users\madhan.kumar\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PySparkPractice")
    .getOrCreate()
)

print(spark.version)

c:\Users\madhan.kumar\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [3]:
data = [
    (1, "Madhan", "Chennai", "Electronics", 50000, "2026-01-10"),
    (2, "Suriya", "Bangalore", "Electronics", 100000, "2026-01-11"),
    (3, "Siva", "Chennai", "Clothing", 5000, "2026-01-12"),
    (4, "Rahul", "Mumbai", "Furniture", 15000, "2026-01-13"),
    (5, "Tejas", "Chennai", "Clothing", 8000, "2026-01-14"),
    (6, "Karthik", "Bangalore", "Electronics", 45000, "2026-01-15"),
    (7, "Pravan", "Mumbai", "Clothing", 7000, "2026-01-16"),
    (8, "Jerom", "Chennai", "Furniture", 20000, "2026-01-17"),
    (9, "Suresh", "Bangalore", "Clothing", 6000, "2026-01-18"),
    (10, "Abhishek", "Mumbai", "Electronics", 55000, "2026-01-19")
]

columns = [
    "order_id",
    "customer",
    "city",
    "category",
    "amount",
    "order_date"
]

In [4]:
df = spark.createDataFrame(data, columns)

In [5]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- order_date: string (nullable = true)



In [6]:
import time

start = time.time()
df.take(5)

before_cache_take = time.time() - start

start = time.time()
df.show(5)

before_cache_show = time.time() - start

+--------+--------+---------+-----------+------+----------+
|order_id|customer|     city|   category|amount|order_date|
+--------+--------+---------+-----------+------+----------+
|       1|  Madhan|  Chennai|Electronics| 50000|2026-01-10|
|       2|  Suriya|Bangalore|Electronics|100000|2026-01-11|
|       3|    Siva|  Chennai|   Clothing|  5000|2026-01-12|
|       4|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|
|       5|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|
+--------+--------+---------+-----------+------+----------+
only showing top 5 rows


In [7]:
df.cache()

DataFrame[order_id: bigint, customer: string, city: string, category: string, amount: bigint, order_date: string]

In [8]:

start = time.time()
df.take(5)

after_cache_take = time.time() - start

start = time.time()
df.show(5)

after_cache_show = time.time() - start

+--------+--------+---------+-----------+------+----------+
|order_id|customer|     city|   category|amount|order_date|
+--------+--------+---------+-----------+------+----------+
|       1|  Madhan|  Chennai|Electronics| 50000|2026-01-10|
|       2|  Suriya|Bangalore|Electronics|100000|2026-01-11|
|       3|    Siva|  Chennai|   Clothing|  5000|2026-01-12|
|       4|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|
|       5|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|
+--------+--------+---------+-----------+------+----------+
only showing top 5 rows


In [9]:
import pandas as pd

take_pct_diff = ((before_cache_take - after_cache_take) / before_cache_take) * 100
show_pct_diff = ((before_cache_show - after_cache_show) / before_cache_show) * 100

compare_df = pd.DataFrame({
    "Before Cache (s)": [before_cache_take, before_cache_show],
    "After Cache (s)": [after_cache_take, after_cache_show],
    "Percentage Faster (%)": [take_pct_diff, show_pct_diff]
}, index=[".take()", ".show()"])

compare_df["Percentage Faster (%)"] = compare_df["Percentage Faster (%)"].round(2)

compare_df

,Before Cache (s),After Cache (s),Percentage Faster (%)
.take(),30.894699,27.504013,10.97
.show(),36.246892,8.450930,76.69


In [10]:
df.show()

+--------+--------+---------+-----------+------+----------+
|order_id|customer|     city|   category|amount|order_date|
+--------+--------+---------+-----------+------+----------+
|       1|  Madhan|  Chennai|Electronics| 50000|2026-01-10|
|       2|  Suriya|Bangalore|Electronics|100000|2026-01-11|
|       3|    Siva|  Chennai|   Clothing|  5000|2026-01-12|
|       4|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|
|       5|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|
|       6| Karthik|Bangalore|Electronics| 45000|2026-01-15|
|       7|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|
|       8|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|
|       9|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|
|      10|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|
+--------+--------+---------+-----------+------+----------+



In [11]:
# Task 1 - Display (customer, city, amount)

df.select(["customer", "city", "amount"]).show()

+--------+---------+------+
|customer|     city|amount|
+--------+---------+------+
|  Madhan|  Chennai| 50000|
|  Suriya|Bangalore|100000|
|    Siva|  Chennai|  5000|
|   Rahul|   Mumbai| 15000|
|   Tejas|  Chennai|  8000|
| Karthik|Bangalore| 45000|
|  Pravan|   Mumbai|  7000|
|   Jerom|  Chennai| 20000|
|  Suresh|Bangalore|  6000|
|Abhishek|   Mumbai| 55000|
+--------+---------+------+



In [12]:
# Task 2 - orders with amount > 20000

df.where(df.amount > 20000).show()

+--------+--------+---------+-----------+------+----------+
|order_id|customer|     city|   category|amount|order_date|
+--------+--------+---------+-----------+------+----------+
|       1|  Madhan|  Chennai|Electronics| 50000|2026-01-10|
|       2|  Suriya|Bangalore|Electronics|100000|2026-01-11|
|       6| Karthik|Bangalore|Electronics| 45000|2026-01-15|
|      10|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|
+--------+--------+---------+-----------+------+----------+



In [13]:
# Task 3 - orders with city = chennai

df.where(df.city == "Chennai").show()

+--------+--------+-------+-----------+------+----------+
|order_id|customer|   city|   category|amount|order_date|
+--------+--------+-------+-----------+------+----------+
|       1|  Madhan|Chennai|Electronics| 50000|2026-01-10|
|       3|    Siva|Chennai|   Clothing|  5000|2026-01-12|
|       5|   Tejas|Chennai|   Clothing|  8000|2026-01-14|
|       8|   Jerom|Chennai|  Furniture| 20000|2026-01-17|
+--------+--------+-------+-----------+------+----------+



In [14]:
# Task 4 - create new column "amount_with_tax"

df = df.withColumn("amount_with_tax", df.amount * 1.18)
df.show()

+--------+--------+---------+-----------+------+----------+---------------+
|order_id|customer|     city|   category|amount|order_date|amount_with_tax|
+--------+--------+---------+-----------+------+----------+---------------+
|       1|  Madhan|  Chennai|Electronics| 50000|2026-01-10|        59000.0|
|       2|  Suriya|Bangalore|Electronics|100000|2026-01-11|       118000.0|
|       3|    Siva|  Chennai|   Clothing|  5000|2026-01-12|         5900.0|
|       4|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|        17700.0|
|       5|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|         9440.0|
|       6| Karthik|Bangalore|Electronics| 45000|2026-01-15|        53100.0|
|       7|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|         8260.0|
|       8|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|        23600.0|
|       9|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|         7080.0|
|      10|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|        64900.0|
+--------+--

In [15]:
# Task 5 - create new column = order_size

import pyspark.sql.functions as F

df= df.withColumn(
    "order_size", 
    F.when(df.amount >= 40000, "High")
     .when(df.amount >= 10000, "Medium")
     .otherwise("Low")
)
df.select(["order_id", "customer", "order_size"]).show()

+--------+--------+----------+
|order_id|customer|order_size|
+--------+--------+----------+
|       1|  Madhan|      High|
|       2|  Suriya|      High|
|       3|    Siva|       Low|
|       4|   Rahul|    Medium|
|       5|   Tejas|       Low|
|       6| Karthik|      High|
|       7|  Pravan|       Low|
|       8|   Jerom|    Medium|
|       9|  Suresh|       Low|
|      10|Abhishek|      High|
+--------+--------+----------+



In [16]:
# Task 6 - total sales

from pyspark.sql.functions import sum

df.agg(sum("amount").alias("total_sales")).show()

+-----------+
|total_sales|
+-----------+
|     311000|
+-----------+



In [17]:
# Task 7 - avg, max, min of amt

from pyspark.sql.functions import avg, max, min

df.agg(
    avg("amount"),
    max("amount"),
    min("amount")
).show()

+-----------+-----------+-----------+
|avg(amount)|max(amount)|min(amount)|
+-----------+-----------+-----------+
|    31100.0|     100000|       5000|
+-----------+-----------+-----------+



In [18]:
# Task 8 - total sales by city

# df.groupBy("city").sum("amount").alias("total_sales").show() 

# or

# (df.groupBy("city")
#    .sum("amount")
#    .withColumnRenamed("sum(amount)", "total_sales")
# ).show()

# or
df.groupBy("city").agg(
    sum("amount").alias("total_sales")
).show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|  Chennai|      83000|
|Bangalore|     151000|
|   Mumbai|      77000|
+---------+-----------+



In [19]:
# Task 9 - Total sales by category

df.groupBy("category").agg(
    sum("amount").alias("total_sales")
).show()

+-----------+-----------+
|   category|total_sales|
+-----------+-----------+
|Electronics|     250000|
|   Clothing|      26000|
|  Furniture|      35000|
+-----------+-----------+



In [20]:
# Task 10 - no. of orders by city

from pyspark.sql.functions import count

df.groupBy("city").agg(
    count("order_id").alias("Number of Orders/city")
).show()

+---------+---------------------+
|     city|Number of Orders/city|
+---------+---------------------+
|  Chennai|                    4|
|Bangalore|                    3|
|   Mumbai|                    3|
+---------+---------------------+



In [21]:
# Task 11 - avg order amt by category

(df.groupBy("category")
   .avg("amount")
   .withColumnRenamed("avg(amount)", "avg_sales")
).show()

+-----------+---------+
|   category|avg_sales|
+-----------+---------+
|Electronics|  62500.0|
|   Clothing|   6500.0|
|  Furniture|  17500.0|
+-----------+---------+



In [22]:
# Task 12 - highest total sales by city

(df.groupBy("city")
   .sum("amount")
   .sort("sum(amount)", ascending=False)
   .withColumnRenamed("sum(amount)", "total_sales")
).limit(1).show()

+---------+-----------+
|     city|total_sales|
+---------+-----------+
|Bangalore|     151000|
+---------+-----------+



In [23]:
# Task 13 - for every city (total, avg, max, min, no.)

(df.groupBy("city")
   .agg(
       sum("amount").alias("total"),
       F.round(avg("amount"),2).alias("avg"),
       max("amount").alias("max"),
       min("amount").alias("min"),
       count("amount").alias("no_of_orders")
   )
).show()

+---------+------+--------+------+----+------------+
|     city| total|     avg|   max| min|no_of_orders|
+---------+------+--------+------+----+------------+
|  Chennai| 83000| 20750.0| 50000|5000|           4|
|Bangalore|151000|50333.33|100000|6000|           3|
|   Mumbai| 77000|25666.67| 55000|7000|           3|
+---------+------+--------+------+----+------------+



In [24]:
# Task 14 - for every category (revenue, avg order value, no. of orders)

(df.groupBy("city")
   .agg(
       sum("amount").alias("total"),
       F.round(avg("amount"),2).alias("avg"),
       max("amount").alias("max"),
       min("amount").alias("min"),
       count("amount").alias("no_of_orders")
   )
).show()

+---------+------+--------+------+----+------------+
|     city| total|     avg|   max| min|no_of_orders|
+---------+------+--------+------+----+------------+
|  Chennai| 83000| 20750.0| 50000|5000|           4|
|Bangalore|151000|50333.33|100000|6000|           3|
|   Mumbai| 77000|25666.67| 55000|7000|           3|
+---------+------+--------+------+----+------------+



In [25]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- order_date: string (nullable = true)
 |-- amount_with_tax: double (nullable = true)
 |-- order_size: string (nullable = false)



In [26]:
# Task 15 - create DF with strucktype and structfield

from datetime import date
from pyspark.sql.types import *

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer", StringType(), True),
    StructField("city", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("order_date", StringType(), True)
])

df = spark.createDataFrame(data, schema=schema)

spark.catalog.clearCache()

In [27]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- order_date: string (nullable = true)



In [28]:
# Task 16 - convert order_date String -> date

df = df.withColumn(
    "order_date",
    F.to_date("order_date")
)

df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- order_date: date (nullable = true)



**Note:** `cache()` is lazy, so I cleared the cache and cached the new DataFrame without triggering an action like `count()`, causing the first execution to still take time.


In [29]:
df.cache()
df.count()

10

In [30]:
df.show()

+--------+--------+---------+-----------+------+----------+
|order_id|customer|     city|   category|amount|order_date|
+--------+--------+---------+-----------+------+----------+
|       1|  Madhan|  Chennai|Electronics| 50000|2026-01-10|
|       2|  Suriya|Bangalore|Electronics|100000|2026-01-11|
|       3|    Siva|  Chennai|   Clothing|  5000|2026-01-12|
|       4|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|
|       5|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|
|       6| Karthik|Bangalore|Electronics| 45000|2026-01-15|
|       7|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|
|       8|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|
|       9|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|
|      10|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|
+--------+--------+---------+-----------+------+----------+



In [31]:
# Task 17 - year month day

df.select(
    F.year(df.order_date).alias("Year"),
    F.month(df.order_date).alias("Month"),
    F.dayofmonth(df.order_date).alias("Day")
).show()

+----+-----+---+
|Year|Month|Day|
+----+-----+---+
|2026|    1| 10|
|2026|    1| 11|
|2026|    1| 12|
|2026|    1| 13|
|2026|    1| 14|
|2026|    1| 15|
|2026|    1| 16|
|2026|    1| 17|
|2026|    1| 18|
|2026|    1| 19|
+----+-----+---+



In [32]:
dirty_data = [
    (11, "Ravi", "Chennai", "Electronics", 25000, "2026-01-20"),
    (12, None, "Mumbai", "Clothing", 9000, "2026-01-21"),
    (13, "Kumar", None, "Furniture", 12000, "2026-01-22"),
    (14, "Ravi", "Chennai", "Electronics", 25000, "2026-01-20"),
    (15, "Ravi", "Chennai", "Electronics", 25000, "2026-01-20"),
    (16, "Arun", "Bangalore", "Furniture", 18000, "2026-01-23"),
    (17, "Arun", "Bangalore", "Furniture", 18000, "2026-01-23"),
]

In [33]:
dirty_df = spark.createDataFrame(dirty_data, schema=schema)
dirty_df.withColumn(
    "order_date",
    F.to_date("order_date")
)

DataFrame[order_id: int, customer: string, city: string, category: string, amount: int, order_date: date]

In [34]:
df = df.union(dirty_df)

In [35]:
df.cache()
df.count()

17

In [36]:
df.show()

+--------+--------+---------+-----------+------+----------+
|order_id|customer|     city|   category|amount|order_date|
+--------+--------+---------+-----------+------+----------+
|       1|  Madhan|  Chennai|Electronics| 50000|2026-01-10|
|       2|  Suriya|Bangalore|Electronics|100000|2026-01-11|
|       3|    Siva|  Chennai|   Clothing|  5000|2026-01-12|
|       4|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|
|       5|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|
|       6| Karthik|Bangalore|Electronics| 45000|2026-01-15|
|       7|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|
|       8|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|
|       9|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|
|      10|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|
|      11|    Ravi|  Chennai|Electronics| 25000|2026-01-20|
|      12|    NULL|   Mumbai|   Clothing|  9000|2026-01-21|
|      13|   Kumar|     NULL|  Furniture| 12000|2026-01-22|
|      14|    Ravi|  Chennai|Electronics

In [37]:
# Task 18 - find null values

cols = df.columns

df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in cols
]).show()

+--------+--------+----+--------+------+----------+
|order_id|customer|city|category|amount|order_date|
+--------+--------+----+--------+------+----------+
|       0|       1|   1|       0|     0|         0|
+--------+--------+----+--------+------+----------+



In [38]:
# Task 19 - remove rows containing null values

df = df.dropna()

In [39]:
df.show()

+--------+--------+---------+-----------+------+----------+
|order_id|customer|     city|   category|amount|order_date|
+--------+--------+---------+-----------+------+----------+
|       1|  Madhan|  Chennai|Electronics| 50000|2026-01-10|
|       2|  Suriya|Bangalore|Electronics|100000|2026-01-11|
|       3|    Siva|  Chennai|   Clothing|  5000|2026-01-12|
|       4|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|
|       5|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|
|       6| Karthik|Bangalore|Electronics| 45000|2026-01-15|
|       7|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|
|       8|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|
|       9|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|
|      10|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|
|      11|    Ravi|  Chennai|Electronics| 25000|2026-01-20|
|      14|    Ravi|  Chennai|Electronics| 25000|2026-01-20|
|      15|    Ravi|  Chennai|Electronics| 25000|2026-01-20|
|      16|    Arun|Bangalore|  Furniture

In [40]:
# Task 20 - replace null values of customer witn "Unknown"

df = df.withColumn(
    "customer",
    F.when(df.customer.isNull(), "Unknown")
     .otherwise(df.customer)
)

In [41]:
df.show()

+--------+--------+---------+-----------+------+----------+
|order_id|customer|     city|   category|amount|order_date|
+--------+--------+---------+-----------+------+----------+
|       1|  Madhan|  Chennai|Electronics| 50000|2026-01-10|
|       2|  Suriya|Bangalore|Electronics|100000|2026-01-11|
|       3|    Siva|  Chennai|   Clothing|  5000|2026-01-12|
|       4|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|
|       5|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|
|       6| Karthik|Bangalore|Electronics| 45000|2026-01-15|
|       7|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|
|       8|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|
|       9|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|
|      10|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|
|      11|    Ravi|  Chennai|Electronics| 25000|2026-01-20|
|      14|    Ravi|  Chennai|Electronics| 25000|2026-01-20|
|      15|    Ravi|  Chennai|Electronics| 25000|2026-01-20|
|      16|    Arun|Bangalore|  Furniture

In [56]:
dup = df.groupBy(
    ["customer", "city", "category", "amount", "order_date"]
).agg(
    count(df.customer).alias("dup_count")
).filter(
    F.col("dup_count") > 1
)

dup = df.join(
    dup, ["customer", "city", "category", "amount", "order_date"], "right"
)

dup = dup.groupBy(["customer", "city", "category", "amount", "order_date"]).agg(
    F.min("order_id").alias("keep_id")
)

dup = df.join(
    dup,
    ["customer", "city", "category", "amount", "order_date"],
    "left"
)

dup = dup.where(
    (F.col("order_id") == F.col("keep_id")) |
    (F.col("keep_id").isNull())
)

dup = dup.drop("keep_id")

dup.show()

+--------+---------+-----------+------+----------+--------+
|customer|     city|   category|amount|order_date|order_id|
+--------+---------+-----------+------+----------+--------+
|  Madhan|  Chennai|Electronics| 50000|2026-01-10|       1|
|  Suriya|Bangalore|Electronics|100000|2026-01-11|       2|
|    Siva|  Chennai|   Clothing|  5000|2026-01-12|       3|
|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|       4|
|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|       5|
| Karthik|Bangalore|Electronics| 45000|2026-01-15|       6|
|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|       7|
|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|       8|
|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|       9|
|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|      10|
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      11|
|    Arun|Bangalore|  Furniture| 18000|2026-01-23|      16|
+--------+---------+-----------+------+----------+--------+



In [57]:
# Task 21 - drop duplicates

# groupBy duplicate columns (not order_id since it can be unique)
# get count more than 1
duplicated_df = df.groupBy(
    ["customer", "city", "category", "amount", "order_date"]
).agg(
    F.count(df.customer).alias("count")
).filter(
    F.col("count") > 1
)

# drop count column
duplicated_df = duplicated_df.drop("count")

In [58]:
duplicated_df.show()

+--------+---------+-----------+------+----------+
|customer|     city|   category|amount|order_date|
+--------+---------+-----------+------+----------+
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|
|    Arun|Bangalore|  Furniture| 18000|2026-01-23|
+--------+---------+-----------+------+----------+



In [59]:
# right join with df and duplicated df
# so we can get the duplicated rows
df.join(duplicated_df, ["customer", "city", "category", "amount", "order_date"], "right").show()

+--------+---------+-----------+------+----------+--------+
|customer|     city|   category|amount|order_date|order_id|
+--------+---------+-----------+------+----------+--------+
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      15|
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      14|
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      11|
|    Arun|Bangalore|  Furniture| 18000|2026-01-23|      17|
|    Arun|Bangalore|  Furniture| 18000|2026-01-23|      16|
+--------+---------+-----------+------+----------+--------+



In [60]:
# get the order_ids to remove
duplicated_order_ids = df.join(duplicated_df, ["customer", "city", "category", "amount", "order_date"], "right").sort("order_id")
duplicated_order_ids.show()

+--------+---------+-----------+------+----------+--------+
|customer|     city|   category|amount|order_date|order_id|
+--------+---------+-----------+------+----------+--------+
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      11|
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      14|
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      15|
|    Arun|Bangalore|  Furniture| 18000|2026-01-23|      16|
|    Arun|Bangalore|  Furniture| 18000|2026-01-23|      17|
+--------+---------+-----------+------+----------+--------+



In [61]:
# it shows what you want to keep
duplicated_order_ids = duplicated_order_ids.groupBy(["customer", "city", "category", "amount", "order_date"]).agg(min("order_id").alias("keep_id"))

In [62]:
duplicated_order_ids.show()

+--------+---------+-----------+------+----------+-------+
|customer|     city|   category|amount|order_date|keep_id|
+--------+---------+-----------+------+----------+-------+
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|     11|
|    Arun|Bangalore|  Furniture| 18000|2026-01-23|     16|
+--------+---------+-----------+------+----------+-------+



In [63]:
# Join with main df, this shows before filter

df.join(duplicated_order_ids, ["customer", "city", "category", "amount", "order_date"], "left").show()

+--------+---------+-----------+------+----------+--------+-------+
|customer|     city|   category|amount|order_date|order_id|keep_id|
+--------+---------+-----------+------+----------+--------+-------+
|  Madhan|  Chennai|Electronics| 50000|2026-01-10|       1|   NULL|
|  Suriya|Bangalore|Electronics|100000|2026-01-11|       2|   NULL|
|    Siva|  Chennai|   Clothing|  5000|2026-01-12|       3|   NULL|
|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|       4|   NULL|
|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|       5|   NULL|
| Karthik|Bangalore|Electronics| 45000|2026-01-15|       6|   NULL|
|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|       7|   NULL|
|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|       8|   NULL|
|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|       9|   NULL|
|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|      10|   NULL|
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      11|     11|
|    Ravi|  Chennai|Electronics| 25000|2026-01-2

In [64]:
# Join with main df

df.join(duplicated_order_ids, ["customer", "city", "category", "amount", "order_date"], "left").filter((F.col("order_id") == F.col("keep_id")) | (F.col("keep_id").isNull())).show()

+--------+---------+-----------+------+----------+--------+-------+
|customer|     city|   category|amount|order_date|order_id|keep_id|
+--------+---------+-----------+------+----------+--------+-------+
|  Madhan|  Chennai|Electronics| 50000|2026-01-10|       1|   NULL|
|  Suriya|Bangalore|Electronics|100000|2026-01-11|       2|   NULL|
|    Siva|  Chennai|   Clothing|  5000|2026-01-12|       3|   NULL|
|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|       4|   NULL|
|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|       5|   NULL|
| Karthik|Bangalore|Electronics| 45000|2026-01-15|       6|   NULL|
|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|       7|   NULL|
|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|       8|   NULL|
|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|       9|   NULL|
|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|      10|   NULL|
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      11|     11|
|    Arun|Bangalore|  Furniture| 18000|2026-01-2

In [65]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- order_date: date (nullable = true)



In [66]:
df = df.join(
    duplicated_order_ids, 
    ["customer", "city", "category", "amount", "order_date"], 
    "left") \
    .filter(
        (F.col("order_id") == F.col("keep_id")) | 
        (F.col("keep_id").isNull()
    )
)

df = df.drop("keep_id")

df.show()

+--------+---------+-----------+------+----------+--------+
|customer|     city|   category|amount|order_date|order_id|
+--------+---------+-----------+------+----------+--------+
|  Madhan|  Chennai|Electronics| 50000|2026-01-10|       1|
|  Suriya|Bangalore|Electronics|100000|2026-01-11|       2|
|    Siva|  Chennai|   Clothing|  5000|2026-01-12|       3|
|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|       4|
|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|       5|
| Karthik|Bangalore|Electronics| 45000|2026-01-15|       6|
|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|       7|
|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|       8|
|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|       9|
|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|      10|
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      11|
|    Arun|Bangalore|  Furniture| 18000|2026-01-23|      16|
+--------+---------+-----------+------+----------+--------+



In [67]:
customer_data = [
    (1, "Madhan", "Gold"),
    (2, "Arun", "Silver"),
    (3, "Priya", "Gold"),
    (4, "Rahul", "Bronze"),
    (5, "Divya", "Gold"),
]

customer_df = spark.createDataFrame(
    customer_data,
    ["customer_id", "customer", "membership"]
)

customer_df.cache()
customer_df.count()

5

In [68]:
from pyspark.sql import Window

window = Window.orderBy("order_id")

df = df.withColumn("customer_id", F.row_number().over(window))

df.show()

+--------+---------+-----------+------+----------+--------+-----------+
|customer|     city|   category|amount|order_date|order_id|customer_id|
+--------+---------+-----------+------+----------+--------+-----------+
|  Madhan|  Chennai|Electronics| 50000|2026-01-10|       1|          1|
|  Suriya|Bangalore|Electronics|100000|2026-01-11|       2|          2|
|    Siva|  Chennai|   Clothing|  5000|2026-01-12|       3|          3|
|   Rahul|   Mumbai|  Furniture| 15000|2026-01-13|       4|          4|
|   Tejas|  Chennai|   Clothing|  8000|2026-01-14|       5|          5|
| Karthik|Bangalore|Electronics| 45000|2026-01-15|       6|          6|
|  Pravan|   Mumbai|   Clothing|  7000|2026-01-16|       7|          7|
|   Jerom|  Chennai|  Furniture| 20000|2026-01-17|       8|          8|
|  Suresh|Bangalore|   Clothing|  6000|2026-01-18|       9|          9|
|Abhishek|   Mumbai|Electronics| 55000|2026-01-19|      10|         10|
|    Ravi|  Chennai|Electronics| 25000|2026-01-20|      11|     

In [69]:
# Task 22 - inner join

df.join(
    customer_df,
    ["customer_id","customer"],
    "inner"
).select(
    "customer_id",
    "order_id",
    "customer"
).show()

+-----------+--------+--------+
|customer_id|order_id|customer|
+-----------+--------+--------+
|          1|       1|  Madhan|
|          4|       4|   Rahul|
+-----------+--------+--------+



In [70]:
# Task 23 - joins

df.join(
    customer_df,
    ["customer_id","customer"],
    "left"
).select(
    "customer_id",
    "order_id",
    "customer"
).show()

df.join(
    customer_df,
    ["customer_id","customer"],
    "right"
).select(
    "customer_id",
    "order_id",
    "customer"
).show()

df.join(
    customer_df,
    ["customer_id","customer"],
    "full"
).select(
    "customer_id",
    "order_id",
    "customer"
).show()

+-----------+--------+--------+
|customer_id|order_id|customer|
+-----------+--------+--------+
|          1|       1|  Madhan|
|          2|       2|  Suriya|
|          3|       3|    Siva|
|          4|       4|   Rahul|
|          5|       5|   Tejas|
|          6|       6| Karthik|
|          7|       7|  Pravan|
|          8|       8|   Jerom|
|          9|       9|  Suresh|
|         10|      10|Abhishek|
|         11|      11|    Ravi|
|         12|      16|    Arun|
+-----------+--------+--------+

+-----------+--------+--------+
|customer_id|order_id|customer|
+-----------+--------+--------+
|          1|       1|  Madhan|
|          2|    NULL|    Arun|
|          3|    NULL|   Priya|
|          4|       4|   Rahul|
|          5|    NULL|   Divya|
+-----------+--------+--------+

+-----------+--------+--------+
|customer_id|order_id|customer|
+-----------+--------+--------+
|          1|       1|  Madhan|
|          2|    NULL|    Arun|
|          2|       2|  Suriya|
|     

In [71]:
data = [
    ("Arun",   "Chennai",   500),
    ("Arun",   "Chennai",   800),
    ("Arun",   "Chennai",   300),
    ("Bala",   "Bangalore", 700),
    ("Bala",   "Bangalore", 400),
    ("Bala",   "Bangalore", 900),
    ("Kumar",  "Chennai",   600),
    ("Kumar",  "Chennai",   200),
    ("Kumar",  "Chennai",   1000),
    ("Divya",  "Mumbai",    300),
    ("Divya",  "Mumbai",    900),
    ("Divya",  "Mumbai",    500),
    ("Priya",  "Bangalore", 800),
    ("Priya",  "Bangalore", 600),
    ("Priya",  "Bangalore", 200)
]

columns = ["customer", "city", "amount"]

df = spark.createDataFrame(data, columns)

df.cache()
df.show()

+--------+---------+------+
|customer|     city|amount|
+--------+---------+------+
|    Arun|  Chennai|   500|
|    Arun|  Chennai|   800|
|    Arun|  Chennai|   300|
|    Bala|Bangalore|   700|
|    Bala|Bangalore|   400|
|    Bala|Bangalore|   900|
|   Kumar|  Chennai|   600|
|   Kumar|  Chennai|   200|
|   Kumar|  Chennai|  1000|
|   Divya|   Mumbai|   300|
|   Divya|   Mumbai|   900|
|   Divya|   Mumbai|   500|
|   Priya|Bangalore|   800|
|   Priya|Bangalore|   600|
|   Priya|Bangalore|   200|
+--------+---------+------+



In [72]:
# Task 24 - highest value order for each city

window = Window.partitionBy("city").orderBy(F.col("amount").desc())

highest_value = df.withColumn(
    "row_num",
    F.row_number().over(window)
)

highest_value.show()

+--------+---------+------+-------+
|customer|     city|amount|row_num|
+--------+---------+------+-------+
|    Bala|Bangalore|   900|      1|
|   Priya|Bangalore|   800|      2|
|    Bala|Bangalore|   700|      3|
|   Priya|Bangalore|   600|      4|
|    Bala|Bangalore|   400|      5|
|   Priya|Bangalore|   200|      6|
|   Kumar|  Chennai|  1000|      1|
|    Arun|  Chennai|   800|      2|
|   Kumar|  Chennai|   600|      3|
|    Arun|  Chennai|   500|      4|
|    Arun|  Chennai|   300|      5|
|   Kumar|  Chennai|   200|      6|
|   Divya|   Mumbai|   900|      1|
|   Divya|   Mumbai|   500|      2|
|   Divya|   Mumbai|   300|      3|
+--------+---------+------+-------+



In [73]:
# Task 25 - rank by spend

window = Window.partitionBy(F.col("customer"))

grouped = df.withColumn(
    "sum_amount",
    F.sum("amount").over(window)
)

window = Window.orderBy(F.col("sum_amount").desc())

grouped = grouped.withColumn(
    "rank",
    F.dense_rank().over(window)
)

grouped.show()

+--------+---------+------+----------+----+
|customer|     city|amount|sum_amount|rank|
+--------+---------+------+----------+----+
|    Bala|Bangalore|   700|      2000|   1|
|    Bala|Bangalore|   400|      2000|   1|
|    Bala|Bangalore|   900|      2000|   1|
|   Kumar|  Chennai|   600|      1800|   2|
|   Kumar|  Chennai|   200|      1800|   2|
|   Kumar|  Chennai|  1000|      1800|   2|
|   Divya|   Mumbai|   300|      1700|   3|
|   Divya|   Mumbai|   900|      1700|   3|
|   Divya|   Mumbai|   500|      1700|   3|
|    Arun|  Chennai|   500|      1600|   4|
|    Arun|  Chennai|   800|      1600|   4|
|    Arun|  Chennai|   300|      1600|   4|
|   Priya|Bangalore|   800|      1600|   4|
|   Priya|Bangalore|   600|      1600|   4|
|   Priya|Bangalore|   200|      1600|   4|
+--------+---------+------+----------+----+



In [74]:
rdd = spark.sparkContext.parallelize(
    [10, 20, 30, 40, 50]
)

In [75]:
# Task 26 - rdd map

result = rdd.map(lambda x: x * 2)

result.collect()

[20, 40, 60, 80, 100]

In [76]:
# Task 27 - rdd filter

result = rdd.filter(lambda x: x > 25)

result.collect()

[30, 40, 50]

In [77]:
# Task 28 - rdd reduce

result = rdd.reduce(lambda x, y: x + y)

result

150

In [78]:
# Task 29 - rdd map squares

result = rdd.map(lambda x: x ** 2)

result.collect()

[100, 400, 900, 1600, 2500]

In [79]:
words = [
    "spark",
    "python",
    "spark",
    "hadoop",
    "python",
    "spark"
]

rdd = spark.sparkContext.parallelize(words)

result = rdd.map(
    lambda word: (word, 1)
).reduceByKey(
    lambda x, y: x + y
)

result.collect()

[('python', 2), ('hadoop', 1), ('spark', 3)]

In [ ]:
# Task 33 - Dataframe -> rdd

rdd_df = df.rdd.map(
    lambda row: (row.customer, row.amount)
)

rdd_df.collect()

[('Arun', 500),
 ('Arun', 800),
 ('Arun', 300),
 ('Bala', 700),
 ('Bala', 400),
 ('Bala', 900),
 ('Kumar', 600),
 ('Kumar', 200),
 ('Kumar', 1000),
 ('Divya', 300),
 ('Divya', 900),
 ('Divya', 500),
 ('Priya', 800),
 ('Priya', 600),
 ('Priya', 200)]

In [86]:
# Task 34 - rdd -> Dataframe

schema = StructType([
    StructField("name", StringType()),
    StructField("amount", IntegerType())
])

df_rdd = rdd_df.toDF(schema=schema)

df_rdd.show()

+-----+------+
| name|amount|
+-----+------+
| Arun|   500|
| Arun|   800|
| Arun|   300|
| Bala|   700|
| Bala|   400|
| Bala|   900|
|Kumar|   600|
|Kumar|   200|
|Kumar|  1000|
|Divya|   300|
|Divya|   900|
|Divya|   500|
|Priya|   800|
|Priya|   600|
|Priya|   200|
+-----+------+



In [88]:
df.createOrReplaceTempView("orders")

In [89]:
# Task 35 - sql orders > 20000

spark.sql("""
    SELECT * FROM
    orders
    WHERE amount > 20000
""").show()

+--------+----+------+
|customer|city|amount|
+--------+----+------+
+--------+----+------+



In [92]:
# Task 36 - total sales by city

spark.sql("""
    SELECT 
        city, 
        SUM(amount) as sum_amount 
    FROM
    orders
    GROUP BY city
""").show()

+---------+----------+
|     city|sum_amount|
+---------+----------+
|  Chennai|      3400|
|Bangalore|      3600|
|   Mumbai|      1700|
+---------+----------+



In [95]:
# Task 37 - average sales by category

spark.sql("""
    SELECT 
        city, 
        ROUND(AVG(amount), 2) as sum_amount 
    FROM
    orders
    GROUP BY city
""").show()

+---------+----------+
|     city|sum_amount|
+---------+----------+
|  Chennai|    566.67|
|Bangalore|     600.0|
|   Mumbai|    566.67|
+---------+----------+

